# 5.5 What is the Fourier transform doing?

:::{margin}
The intuition presented here was largely inspired by [this excellent 3Blue1Brown video](https://www.youtube.com/watch?v=spUNpyF58BY).
:::

The definition of the Fourier transform above can feel like it appears out of nowhere. That is okay. Let us unpack it more intuitively.

How can a single integral possibly pick out the amount of one specific frequency $\omega$ hiding inside an arbitrary signal? Look again at the transform, $X(\omega) = \int x(t)\, e^{-j\omega t}\, dt$, and read it in three steps:

1. **Synthesize a phasor at $-\omega$.** The term $e^{-j\omega t}$ is a complex sinusoid rotating at frequency $\omega$, just in the clockwise direction (the minus sign reverses the direction of rotation).
1. **Multiply to measure similarity.** Multiplying the signal $x(t)$ by this phasor "winds" the signal around the complex plane at rate $\omega$. Wherever the signal's own oscillation matches the winding rate, the product reinforces in a consistent direction.
1. **Integrate to sum over time.** The integral adds up the wound signal across all time, accumulating that reinforcement (or lack of it) into a single complex number.

Intuitively, we are measuring the _correlation_ between $x(t)$ and a phasor probing at frequency $\omega$. The cleanest way to see this is to look at the wound-up signal in the complex plane and track its **center of mass** (the average of all the wound points). When the probe frequency matches a frequency present in the signal, the winding lines up and the center of mass is pulled far from the origin, yielding a large $|X(\omega)|$. When the probe frequency does not match, the winding smears symmetrically around the origin, the contributions cancel, and the center of mass sits near zero:

:::{figure}
![Three complex-plane plots of the same signal wound at different probe frequencies. At a probe of 2 Hz the curve forms balanced lobes and the center of mass sits at the origin. At 3 Hz, matching the signal, the curve bunches to one side and the center of mass is pulled well away from the origin. At 4 Hz the center of mass returns to near the origin.](./assets/fig-ft-intuition.png)

Winding a signal (here a 3 Hz oscillation) around the complex plane at three probe frequencies. The red dot is the center of mass. Only at the matching probe of 3 Hz (middle) is the center of mass pulled away from the origin, signaling a large amplitude at that frequency. At non-matching probes (2 and 4 Hz), the contributions cancel and the center of mass stays near zero.
:::

The center of mass is an average, and the integral in the Fourier transform is a (continuous) sum, so the two are proportional. The Fourier transform sweeps this probe across every frequency $\omega$ and records, for each one, how far off-origin the center of mass lands.

In [ ]:
import numpy as np
from manim import *

import icm_anim as anim

In [ ]:
class FourierWinding(Scene):
    """The winding machine as film: sweep the probe frequency g from 0.5 to
    6 Hz and watch the center of mass discover the 2 Hz and 4 Hz tones."""

    def construct(self):
        T = 2.0
        x2 = lambda t: np.sin(TAU * 2 * t)
        x4 = lambda t: 0.5 * np.sin(TAU * 4 * t)
        x = lambda t: x2(t) + x4(t)

        # Center-of-mass paths on a dense probe grid, interpolated per frame.
        ts = np.linspace(0.0, T, 400, endpoint=False)
        gs = np.linspace(0.5, 5.0, 451)
        ph = np.exp(-1j * TAU * np.outer(gs, ts))
        com2_g = (x2(ts)[None, :] * ph).mean(1)
        com4_g = (x4(ts)[None, :] * ph).mean(1)
        mag2_g, mag4_g = np.abs(com2_g), np.abs(com4_g)
        mag_g = np.abs(com2_g + com4_g)

        def com(arr, gv):
            return np.interp(gv, gs, arr.real) + 1j * np.interp(gv, gs, arr.imag)

        def vec(c):
            return np.array([c.real, c.imag, 0.0])

        g = ValueTracker(gs[0])

        # --- Winding plane (left): unit circle, wound components, com ------
        hub = LEFT * 4.6 + DOWN * 0.65
        S = 1.95  # scene units per unit of |w|
        circle = Circle(radius=S, color=anim.IRON, stroke_width=1.2).move_to(hub)
        cross = VGroup(
            Line(hub + LEFT * (S + 0.15), hub + RIGHT * (S + 0.15), color=anim.IRON, stroke_width=0.8),
            Line(hub + DOWN * (S + 0.15), hub + UP * (S + 0.15), color=anim.IRON, stroke_width=0.8),
        )

        def wound(xf, color):
            return always_redraw(lambda: ParametricFunction(
                lambda t: hub + S * xf(t) * np.array([
                    np.cos(-TAU * g.get_value() * t),
                    np.sin(-TAU * g.get_value() * t), 0.0,
                ]),
                t_range=[0, T, 0.004],
                color=color, stroke_width=2.2, stroke_opacity=0.55,
            ))

        w2 = wound(x2, anim.BLUE)
        w4 = wound(x4, anim.TEAL)

        # com contributions tip-to-tail: blue 2 Hz arrow, teal 4 Hz on top
        arrow2 = always_redraw(lambda: Arrow(
            hub, hub + S * vec(com(com2_g, g.get_value())),
            buff=0, color=anim.BLUE, stroke_width=4.5,
        ))
        arrow4 = always_redraw(lambda: Arrow(
            hub + S * vec(com(com2_g, g.get_value())),
            hub + S * vec(com(com2_g, g.get_value()) + com(com4_g, g.get_value())),
            buff=0, color=anim.TEAL, stroke_width=4.5,
        ))
        com_dot = always_redraw(lambda: Dot(
            hub + S * vec(com(com2_g, g.get_value()) + com(com4_g, g.get_value())),
            color=anim.RED, radius=0.08,
        ))

        readout_label = MathTex(r"\text{probe}\ g =")
        readout_num = DecimalNumber(gs[0], num_decimal_places=2)
        readout_hz = MathTex(r"\text{Hz}")
        readout = VGroup(readout_label, readout_num, readout_hz)
        readout.arrange(RIGHT, buff=0.18).scale(0.7).move_to(hub + UP * (S + 0.6))
        readout_num.add_updater(lambda m: m.set_value(g.get_value()))

        # --- Signal panel (top right): tones, composite, shrinking probe ---
        sig = Axes(
            x_range=[0, T, 1], y_range=[-1.7, 1.7, 1],
            x_length=6.0, y_length=1.9,
            axis_config={"color": anim.IRON, "include_ticks": False,
                         "stroke_width": 1.5},
            tips=False,
        ).move_to(RIGHT * 3.0 + UP * 1.5)
        sig_x2 = sig.plot(x2, x_range=[0, T, 0.005], color=anim.BLUE,
                          stroke_width=1.6, stroke_opacity=0.5)
        sig_x4 = sig.plot(x4, x_range=[0, T, 0.005], color=anim.TEAL,
                          stroke_width=1.6, stroke_opacity=0.5)
        sig_x = sig.plot(x, x_range=[0, T, 0.005], color=anim.IRON, stroke_width=3)
        probe = always_redraw(lambda: DashedVMobject(sig.plot(
            lambda t: np.sin(TAU * g.get_value() * t),
            x_range=[0, T, 0.004], color=anim.GOLD, stroke_width=2.2,
        ), num_dashes=80))
        probe_label = MathTex(r"\text{probe}\ \sin(2\pi g\,t)", color=anim.GOLD)
        probe_label.scale(0.5).next_to(sig, UP, buff=0.15).align_to(sig, RIGHT)
        sig_xlabel = MathTex(r"\text{time (s)}").scale(0.45).next_to(sig, DOWN, buff=0.12)

        # --- Spectrum panel (bottom right): |com| grows in as g sweeps -----
        ymax = float(mag_g.max()) * 1.25
        spec = Axes(
            x_range=[0, 5, 1], y_range=[0, ymax, ymax],
            x_length=6.0, y_length=2.2,
            axis_config={"color": anim.IRON, "include_ticks": False,
                         "stroke_width": 1.5},
            tips=False,
        ).move_to(RIGHT * 3.0 + DOWN * 2.05)

        def partial(arr, color, width, opacity=1.0):
            return always_redraw(lambda: spec.plot(
                lambda xx: np.interp(xx, gs, arr),
                x_range=[gs[0], max(g.get_value(), gs[0] + 0.02), 0.01],
                color=color, stroke_width=width, stroke_opacity=opacity,
            ))

        spec_2 = partial(mag2_g, anim.BLUE, 1.6, 0.5)
        spec_4 = partial(mag4_g, anim.TEAL, 1.6, 0.5)
        spec_x = partial(mag_g, anim.IRON, 3)
        spec_dot = always_redraw(lambda: Dot(
            spec.c2p(g.get_value(), np.interp(g.get_value(), gs, mag_g)),
            color=anim.RED, radius=0.065,
        ))
        spec_xlabel = MathTex(r"\text{probe frequency}\ g\ \text{(Hz)}")
        spec_xlabel.scale(0.5).next_to(spec, DOWN, buff=0.15)
        spec_ylabel = MathTex(r"|\text{center of mass}|")
        spec_ylabel.scale(0.5).rotate(PI / 2).next_to(spec, LEFT, buff=0.18)

        title = MathTex(
            r"x(t) =", r"\sin(2\pi\,2\,t)", r"+", r"\tfrac{1}{2}\sin(2\pi\,4\,t)"
        ).scale(0.9).to_edge(UP, buff=0.25)
        title[1].set_color(anim.BLUE)
        title[3].set_color(anim.TEAL)

        def star_at(f_hz, color, text):
            k = int(np.argmin(np.abs(gs - f_hz)))
            lo, hi = max(0, k - 10), min(len(gs), k + 11)
            kk = lo + int(np.argmax(mag_g[lo:hi]))
            point = spec.c2p(gs[kk], mag_g[kk])
            star = Star(5, outer_radius=0.12, inner_radius=0.055,
                        color=color, fill_opacity=1.0).move_to(point + UP * 0.22)
            label = MathTex(text, color=color).scale(0.55).next_to(star, UP, buff=0.08)
            return VGroup(star, label)

        star2 = star_at(2.0, anim.BLUE, r"2\ \text{Hz}")
        star4 = star_at(4.0, anim.TEAL, r"4\ \text{Hz}")

        # --- Film: build the machine, then sweep, pausing at each discovery
        self.play(
            FadeIn(circle, cross, sig, sig_x2, sig_x4, sig_x, spec, readout),
            Write(title),
            FadeIn(probe_label, sig_xlabel, spec_xlabel, spec_ylabel),
            run_time=1.2,
        )
        self.add(w2, w4, arrow2, arrow4, com_dot, probe, spec_2, spec_4, spec_x, spec_dot)
        self.play(g.animate.set_value(2.0), run_time=3.5, rate_func=linear)
        self.play(FadeIn(star2, scale=1.6), run_time=0.5)
        self.wait(0.4)
        self.play(g.animate.set_value(4.0), run_time=4.0, rate_func=linear)
        self.play(FadeIn(star4, scale=1.6), run_time=0.5)
        self.wait(0.4)
        self.play(g.animate.set_value(5.0), run_time=2.0, rate_func=linear)
        self.wait(0.8)


anim.show(FourierWinding)